# 监控与可观测性 第3周:日志与链路追踪

> **学习目标**:理解结构化日志、搭建 Loki 日志聚合、用 OpenTelemetry + Jaeger 实现分布式追踪

---

## Day 15:从 printf 到结构化日志

传统日志(不好):
```
INFO: [2024-01-15 10:30:45] User 123 logged in from 192.168.1.1
```
问题:无法按 user_id 过滤,无法聚合统计,grep 效率低

结构化日志(好):
```json
{"timestamp": "2024-01-15T10:30:45", "level": "INFO", "message": "user_login",
 "user_id": "123", "ip": "192.168.1.1", "duration_ms": 45, "trace_id": "a1b2c3"}
```

好处:可由 Loki/ELK 索引和查询,支持聚合、过滤、关联

## Day 16:Loki 架构

### Loki 的设计哲学

| | Elasticsearch | Loki |
|------|---------------|------|
| 索引策略 | 索引日志全文 → 成本高 | 只索引 label → 低成本 |
| 存储 | 自建或云盘 | 对象存储(S3/GCS) |
| 查询语言 | Lucene 语法 | LogQL(像 PromQL) |
| K8s 友好度 | 中 | 极高(Promtail DaemonSet) |

架构:Container stdout → Promtail (DaemonSet) → Loki → Grafana(同一 UI 查看日志和指标)

## Day 17:LogQL 查询

In [ ]:
queries = {
    "过滤 ERROR": '{app="web"} |= "ERROR"',
    "排除 DEBUG": '{app="web"} != "DEBUG"',
    "正则匹配": '{app="web"} |~ "(?i)error|fail"',
    "JSON 解析": '{app="web"} | json | status_code="500"',
    "按级别计数": 'sum(count_over_time({app="web"} | json | level="error" [5m])) by (endpoint)',
    "错误速率": 'rate({app="web"} |= "ERROR" [5m])',
    "关联 trace_id": '{app="web"} |= "a1b2c3d4e5f6"',
}

print("LogQL 常用查询:")
for name, query in queries.items():
    print(f"  {name:12s}: {query}")

## Day 18:分布式追踪概念

```
Trace ID: abc123
│
├── Span A: GET /api/users  [100ms]
│   ├── Span B: auth.verify_token  [5ms]
│   ├── Span C: db.query("SELECT ...")  [60ms]
│   │   ├── Span D: db.connect  [3ms]
│   │   └── Span E: db.execute  [55ms]
│   └── Span F: cache.get("users:list")  [2ms]
│
└── Span G: render_response  [10ms]
```

每个 Span 包含: trace_id(属于哪个 Trace), span_id, parent_span_id, name(操作名), start_time / duration, attributes(自定义标签), status(OK/Error)

## Day 19:OpenTelemetry + Jaeger

In [ ]:
print("OpenTelemetry Python SDK 核心用法")
print("=" * 50)

print("""
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter
from opentelemetry.instrumentation.flask import FlaskInstrumentor

# 1. 初始化
provider = TracerProvider()
exporter = OTLPSpanExporter(endpoint="http://jaeger:4317", insecure=True)
provider.add_span_processor(BatchSpanProcessor(exporter))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer(__name__)

# 2. 自动插桩 (Flask, requests, sqlalchemy 等)
FlaskInstrumentor().instrument_app(app)

# 3. 手动创建 Span
with tracer.start_as_current_span("business_logic") as span:
    span.set_attribute("user.id", user_id)
    span.set_attribute("order.count", len(orders))
    result = process_orders(orders)

# 4. 记录事件
with tracer.start_as_current_span("payment") as span:
    span.add_event("payment_started", {"provider": "stripe"})
    charge = stripe.charge(amount, currency)
    span.add_event("payment_completed", {"charge_id": charge.id})
""")

print("关键:OpenTelemetry 是 CNCF 标准,厂商中立,多语言统一 API")

## Day 20:三个支柱打通

```
场景:用户反馈 /api/orders 很慢

1. Metrics(发现问题)
   在 Grafana 看到 p99 延迟从 200ms 飙升到 3s
   → 时间: 14:30 - 14:45

2. Traces(定位环节)
   从延迟图表通过 Exemplar 跳转到 Jaeger
   → 看到 Trace 显示 "db.query" Span 花了 2.8s
   → trace_id = a1b2c3d4

3. Logs(确认原因)
   在 Grafana 切换到 Loki, 查询: {app="api"} |= "a1b2c3d4"
   → 看到日志: "slow query detected: missing index on orders(created_at)"
   → 修复: CREATE INDEX ON orders(created_at)

完整排查链: Metric → Trace → Log   (< 5 分钟)
传统方式: 30+ 分钟的 grep 和猜测
```

## Day 21:第3周综合练习

In [ ]:
print("=" * 60)
print("第3周综合练习交付清单")
print("=" * 60)

print("""
Python 应用:
  - 结构化日志 (structlog, JSON 格式)
  - OpenTelemetry SDK (自动 + 手动 Span)

Docker Compose 编排:
  ├── Prometheus (指标)
  ├── Grafana (可视化)
  ├── Alertmanager (告警)
  ├── Loki + Promtail (日志聚合)
  ├── Jaeger + OTEL Collector (链路追踪)
  ├── Python Web App
  └── Python Worker App

验证三支柱关联:
  - 注入慢查询故障
  - 在 Grafana 观察到 p99 告警
  - Metric → Exemplar → Jaeger Trace (定位慢查询 Span)
  - Span 的 trace_id → Loki 日志 (看到具体 SQL)
  - 完整排查链: Metric → Trace → Log
""")

print("=" * 60)
print("第3周核心收获:")
print("1. 结构化日志 = JSON + 有意义的字段,让机器可解析")
print("2. Loki 只索引 label 不索引日志内容,低成本 K8s 友好")
print("3. Trace 由 Span 组成树形结构,展示请求的完整调用路径")
print("4. OpenTelemetry 是 CNCF 标准,统一 API,多语言一致")
print("5. 三支柱打通: Metric 发现→Trace 定位→Log 确认")
print("=" * 60)